In [2]:
import pandas as pd

# Cargamos el dataset principal, es decir, el que entrenaremos.
# En nuestro caso, la suma de los dos últimos dígitos de uno de nosotros es 3, por lo que isaremos bank_03.pkl.
try:
    df_entreno = pd.read_pickle("banks/bank_03.pkl")
    print("Datos de entrenamiento cargados correctamente. Dimensiones:", df_entreno.shape)

    # Mostrar las primeras 5 filas para echar un vistazo
    display(df_entreno.head())
except FileNotFoundError:
    print("No se encontró el archivo.")

print("-" * 50)

# Cargamos el dataset de la competición
try:
    df_competicion = pd.read_pickle("banks/bank_competition.pkl")
    print("Datos de la competición cargados correctamente. Dimensiones:", df_competicion.shape)
    display(df_competicion.head())
except FileNotFoundError:
    print("No se encontró el archivo de competición.")

Datos de entrenamiento cargados correctamente. Dimensiones: (11000, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,None,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


--------------------------------------------------
Datos de la competición cargados correctamente. Dimensiones: (162, 16)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
5553,43,management,married,tertiary,no,78,yes,no,cellular,21,nov,36,1,109,1,other
915,34,housemaid,married,secondary,no,0,yes,no,unknown,30,oct,154,1,-1,0,unknown
7652,54,technician,married,secondary,no,3323,yes,yes,cellular,8,apr,59,3,-1,0,unknown
5065,43,blue-collar,single,primary,no,-399,no,yes,cellular,28,jul,662,3,-1,0,unknown
3338,35,blue-collar,married,secondary,no,262,no,no,cellular,15,mar,427,1,181,3,success


In [3]:
import pandas as pd
import numpy as np

# Cargamos el archivo del dataset de entrenamiento y obtenemos filas y columnas.
df = pd.read_pickle("banks/bank_03.pkl")

print("1. FORMA DEL DATASET\n")
print(f"Instancias (filas): {df.shape[0]}")
print(f"Variables (columnas): {df.shape[1]}")

print("\n2. TIPO DE VARIABLES\n")
# Separamos variables numéricas y el resto (categóricas/ordinales)
numericas = df.select_dtypes(include=[np.number]).columns.tolist()
categoricas = df.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Numéricas ({len(numericas)}): {numericas}")
print(f"Categóricas/Ordinales ({len(categoricas)}): {categoricas}")

print("\n3. CATEGÓRICAS CON ALTA CARDINALIDAD (>10 valores únicos)\n")
for col in categoricas:
    n_unicos = df[col].nunique()
    if n_unicos > 10:
        print(f"La variable '{col}' tiene {n_unicos} valores únicos.")

print("\n4. VALORES FALTANTES (Nulos, 'unknown' y 'other')\n")

encontrados = False
# Revisamos todas las columnas
for col in df.columns:
    # Contamos los None
    cantidad_nulos = df[col].isna().sum()
    
    # Contamos cuántos 'unknown' y 'other' hay (solo en texto/categoría)
    cantidad_unknown = 0
    cantidad_other = 0
    if df[col].dtype == 'object' or hasattr(df[col].dtype, 'categories'):
        cantidad_unknown = (df[col] == 'unknown').sum()
        cantidad_other = (df[col] == 'other').sum()
        
    # Sumamos los tres para saber el total de información faltante en esa columna
    total_faltantes = cantidad_nulos + cantidad_unknown + cantidad_other
    
    # Si falta algún dato, lo imprimimos con su desglose
    if total_faltantes > 0:
        porcentaje = (total_faltantes / len(df)) * 100
        
        # Preparamos un texto para saber exactamente qué tipo de faltantes son
        detalles = []
        if cantidad_nulos > 0:
            detalles.append(f"{cantidad_nulos} nulos (None)")
        if cantidad_unknown > 0:
            detalles.append(f"{cantidad_unknown} 'unknown'")
        if cantidad_other > 0:
            detalles.append(f"{cantidad_other} 'other'")
            
        print(f"La variable '{col}' tiene {total_faltantes} valores faltantes ({porcentaje:.2f}%) -> " + " y ".join(detalles))
        encontrados = True

if not encontrados:
    print("No se encontraron valores faltantes, 'unknown' ni 'other'.")

print("\n5. COLUMNAS CONSTANTES O DE ID\n")
constantes_o_id = False
for col in df.columns:
    if df[col].nunique() == 1:
        print(f"Columna constante encontrada: '{col}'")
        constantes_o_id = True
    elif df[col].nunique() == len(df):
        print(f"Columna tipo ID encontrada (todos los valores distintos): '{col}'")
        constantes_o_id = True
if not constantes_o_id:
    print("No se encontraron columnas constantes ni columnas tipo ID.")

print("\n6. TIPO DE PROBLEMA Y BALANCEO\n")
print("Es un problema de CLASIFICACIÓN (predecir 'yes' o 'no' en deposit).")
print("Distribución de la clase objetivo:")
# normalize=True nos da el porcentaje en vez del recuento
print((df['deposit'].value_counts(normalize=True) * 100).round(2))

print("\n7. ANÁLISIS DE LA VARIABLE 'pdays'\n")
porcentaje_pdays_menos1 = (df['pdays'] == -1).mean() * 100
print(f"El {porcentaje_pdays_menos1:.2f}% de los clientes tiene valor -1 en pdays.")
print("Resumen estadístico de pdays:")
print(df['pdays'].describe())

1. FORMA DEL DATASET

Instancias (filas): 11000
Variables (columnas): 17

2. TIPO DE VARIABLES

Numéricas (7): ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Categóricas/Ordinales (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'deposit']

3. CATEGÓRICAS CON ALTA CARDINALIDAD (>10 valores únicos)

La variable 'job' tiene 12 valores únicos.
La variable 'month' tiene 12 valores únicos.

4. VALORES FALTANTES (Nulos, 'unknown' y 'other')

La variable 'job' tiene 68 valores faltantes (0.62%) -> 68 'unknown'
La variable 'marital' tiene 402 valores faltantes (3.65%) -> 402 nulos (None)
La variable 'education' tiene 494 valores faltantes (4.49%) -> 494 'unknown'
La variable 'contact' tiene 2309 valores faltantes (20.99%) -> 2309 'unknown'
La variable 'poutcome' tiene 8733 valores faltantes (79.39%) -> 8205 'unknown' y 528 'other'

5. COLUMNAS CONSTANTES O DE ID

No se encontraron columnas constantes ni columnas tipo ID

In [6]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, pointbiserialr

print("EXTRA: ANÁLISIS ESTADÍSTICO DE DISTRIBUCIÓN Y ESCALADO (PARA KNN)")
# Mapeamos el target a numérico para pruebas estadísticas
if 'deposit_bin' not in df.columns:
    df['deposit_bin'] = df['deposit'].map({'yes': 1, 'no': 0})

for col in numericas:
    # 1. Asimetría (Skewness)
    asimetria = df[col].skew()
    
    # 2. Outliers (Método IQR)
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
    pct_outliers = (outliers / len(df)) * 100
    
    print(f"\n{col}")
    
    # Lógica de decisión automatizada para asimetría
    if abs(asimetria) > 1:
        print(f"  - Asimetría ALTA ({asimetria:.2f}). -> Aplicar transformación logarítmica o de Box-Cox para modelos lineales.")
    elif abs(asimetria) > 0.5:
        print(f"  - Asimetría MEDIA ({asimetria:.2f}).")
    else:
        print(f"  - Bastante Simétrica ({asimetria:.2f}).")
        
    # Lógica de decisión automatizada para escalado (Responde a la sección 4.1 del guion)
    if pct_outliers > 5:
        print(f"  - Outliers: MUCHOS ({pct_outliers:.2f}%). DECISIÓN PARA KNN: Usar RobustScaler.")
    elif pct_outliers > 0:
        print(f"  - Outliers: MODERADOS ({pct_outliers:.2f}%). DECISIÓN PARA KNN: Usar StandardScaler, pero RobustScaler mejor.")
    else:
        print(f"  - Outliers: NINGUNO. DECISIÓN PARA KNN: Usar MinMaxScaler.")


print("\nEXTRA: MULTICOLINEALIDAD (PARA MODELOS LINEALES)")
# Calculamos la correlación de Pearson y buscamos pares altamente correlacionados
corr_matrix = df[numericas].corr().abs()
# Seleccionamos el triángulo superior de la matriz de correlación
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

colineales = [column for column in upper.columns if any(upper[column] > 0.8)]
if len(colineales) > 0:
    print(f"  - ¡Alerta! Alta correlación (>0.8) en: {colineales}")
    print("  -> Eliminar una de las variables del par o aplicar regularización L1 (Lasso).")
else:
    print("  - No se detectó multicolinealidad severa (Pearson < 0.8). Los modelos serán estables.")


print("\nEXTRA: IMPORTANCIA DE ATRIBUTOS")
print("Analizando qué variables tienen mayor poder predictivo sobre 'deposit'...")

print("\n  -- Variables Numéricas (Correlación Punto-Biserial) --")
for col in numericas:
    # Ignoramos pdays aquí porque -1 distorsiona la matemática, lo trataremos aparte
    if col == 'pdays': continue 
    
    corr, p_value = pointbiserialr(df[col], df['deposit_bin'])
    if p_value < 0.05:
        fuerza = "ALTA" if abs(corr) > 0.3 else ("MODERADA" if abs(corr) > 0.1 else "BAJA")
        print(f"  - {col}: Fuerza predictiva (P-value): {fuerza} (r = {corr:.3f})")
    else:
        print(f"  - {col}: NO significativa (p-value >= 0.05). Considerar eliminar para simplificar modelos.")

print("\n  -- Variables Categóricas (Test Chi-Cuadrado) --")
for col in categoricas:
    if col == 'deposit': continue
    
    contingency_table = pd.crosstab(df[col], df['deposit'])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    if p_value < 0.05:
        print(f"  - {col}: Relación significativa con la suscripción (p-value = {p_value:.2e}). Mantener en el modelo.")
    else:
        print(f"  - {col}: NO significativa. La distribución de esta variable es independiente de la suscripción. Podría generar ruido.")


print("\nEXTRA: DECISIÓN DE PREPROCESO AVANZADO PARA 'pdays'")
# Creamos la variable binaria para evaluar matemáticamente su utilidad
df['contactado_previamente'] = (df['pdays'] != -1).astype(int)
contingency_pdays = pd.crosstab(df['contactado_previamente'], df['deposit'])
chi2_pdays, p_pdays, _, _ = chi2_contingency(contingency_pdays)

if p_pdays < 0.05:
    print(f"  - Resultado: Extraer el valor -1 a una nueva variable binaria es ALTAMENTE significativo (p-value = {p_pdays:.2e}).")
    print("  - DECISIÓN PARA PIPELINE: Crear un transformador personalizado que genere esta variable binaria y reemplace los -1 en 'pdays' por la mediana de los contactados o directamente agrupe en bins.")

EXTRA: ANÁLISIS ESTADÍSTICO DE DISTRIBUCIÓN Y ESCALADO (PARA KNN)

age
  - Asimetría MEDIA (0.86).
  - Outliers: MODERADOS (1.55%). DECISIÓN PARA KNN: Usar StandardScaler, pero RobustScaler mejor.

balance
  - Asimetría ALTA (8.25). -> Aplicar transformación logarítmica o de Box-Cox para modelos lineales.
  - Outliers: MUCHOS (9.48%). DECISIÓN PARA KNN: Usar RobustScaler.

day
  - Bastante Simétrica (0.11).
  - Outliers: NINGUNO. DECISIÓN PARA KNN: Usar MinMaxScaler.

duration
  - Asimetría ALTA (2.14). -> Aplicar transformación logarítmica o de Box-Cox para modelos lineales.
  - Outliers: MUCHOS (5.64%). DECISIÓN PARA KNN: Usar RobustScaler.

campaign
  - Asimetría ALTA (5.57). -> Aplicar transformación logarítmica o de Box-Cox para modelos lineales.
  - Outliers: MUCHOS (5.35%). DECISIÓN PARA KNN: Usar RobustScaler.

pdays
  - Asimetría ALTA (2.46). -> Aplicar transformación logarítmica o de Box-Cox para modelos lineales.
  - Outliers: MUCHOS (24.65%). DECISIÓN PARA KNN: Usar RobustS